# 环节 02 · 隔离强度谱系演示

纯 Python 标准库，零依赖。把"选哪一档沙箱"拆成四个可计算的量：

1. 逃逸需要突破的层数（共享内核 vs 独立内核）；
2. 冷启动预算分解（看钱花在哪）；
3. 单机密度（每沙箱固定开销的影响）；
4. 加权选型打分。

> 数字均为**量级估算**，用于排序与直觉，不是性能承诺。

In [ ]:
# §1 六档谱系（量级，非承诺）
TIERS = {
    "进程级(bwrap/Seatbelt)": dict(kernel="共享", layers=1, boot_ms=2,    mem_mb=1,   compat=100),
    "容器(runc)":             dict(kernel="共享", layers=1, boot_ms=300,  mem_mb=10,  compat=100),
    "强化容器(gVisor)":       dict(kernel="共享", layers=2, boot_ms=200,  mem_mb=60,  compat=70),
    "微VM(Firecracker)":      dict(kernel="独立", layers=2, boot_ms=150,  mem_mb=120, compat=95),
    "Wasm(Wasmtime)":         dict(kernel="无",   layers=1, boot_ms=1,    mem_mb=5,   compat=30),
}

print(f"{'档位':<26}{'内核':<6}{'逃逸需穿':<9}{'启动(ms)':<10}{'固定内存(MB)':<13}{'兼容性'}")
print("-" * 82)
for name, t in TIERS.items():
    print(f"{name:<26}{t['kernel']:<6}{t['layers']:<9}{t['boot_ms']:<10}"
          f"{t['mem_mb']:<13}{t['compat']}%")
print()
print("关键分界：内核是否为『独立』—— 决定一次内核 0day 的影响范围")

In [ ]:
# §2 逃逸成本：把"穿层"翻译成攻击者需要的能力
ESCAPE_COST = {
    1: ("内核/运行时漏洞，或配置错误", "中"),
    2: ("Sentry+内核，或 hypervisor+内核（两层都要破）", "高"),
}


def escape_profile(tier):
    t = TIERS[tier]
    desc, level = ESCAPE_COST[t["layers"]]
    return desc, level


for name in TIERS:
    desc, level = escape_profile(name)
    print(f"{name:<26} 逃逸难度={level:<3} 路径={desc}")

## §3 冷启动预算：钱花在哪一段

`总冷启动 = 调度 + 镜像拉取 + 内核 boot + 应用初始化`。
**优化顺序取决于哪一段占比最大**，别照着别人博客乱抄。

In [ ]:
# §3 冷启动预算分解（示例：会话型代码解释器）
BUDGET = [
    ("调度/分配",        120, "平台侧"),
    ("镜像拉取(未缓存)", 800, "可被缓存消除"),
    ("内核 boot",         150, "仅微 VM 档有"),
    ("运行时初始化",      200, "如 Python 启动"),
    ("应用初始化",        900, "如 import pandas"),
]
total = sum(ms for _, ms, _ in BUDGET)
print(f"{'阶段':<20}{'耗时(ms)':<10}{'占比':<8}{'优化手段'}")
print("-" * 62)
for stage, ms, how in BUDGET:
    print(f"{stage:<20}{ms:<10}{ms / total:<8.0%}{how}")
print("-" * 62)
print(f"{'合计':<20}{total:<10}")

app = 900
print()
print(f"应用初始化占比 = {app / total:.0%}")
print("→ 占比高：上快照/预热池收益大；占比低：先优化调度与镜像缓存")

In [ ]:
# §4 单机密度：固定开销怎么吃掉容量
def density(mem_gb, tier, per_task_mb, cpu_cores, task_cores):
    fixed = TIERS[tier]["mem_mb"]
    by_mem = int(mem_gb * 1024 / (fixed + per_task_mb))
    by_cpu = int(cpu_cores / task_cores)
    return min(by_mem, by_cpu), by_mem, by_cpu


MEM_GB = 48
PER_TASK_MB = 300
for tier in ["容器(runc)", "强化容器(gVisor)", "微VM(Firecracker)"]:
    n, m, c = density(MEM_GB, tier, PER_TASK_MB, 12, 0.5)
    print(f"{tier:<24} 上限={n:<4}（内存 {m} / CPU {c}）")
print()
print("注意：内存估的是『固定开销 + 业务峰值』，且要用 P99 峰值而非均值")

In [ ]:
# §5 加权选型打分（按你的威胁模型调权重）
WEIGHTS = {"隔离": 0.4, "启动": 0.2, "密度": 0.2, "兼容": 0.2}

def score(tier):
    t = TIERS[tier]
    iso = min(1.0, t["layers"] / 2)              # 层数越多越强
    boot = 1 - min(t["boot_ms"], 500) / 500      # 越快越高
    dens = 1 - min(t["mem_mb"], 200) / 200
    comp = t["compat"] / 100
    return (iso * WEIGHTS["隔离"] + boot * WEIGHTS["启动"]
            + dens * WEIGHTS["密度"] + comp * WEIGHTS["兼容"])


rank = sorted(TIERS, key=score, reverse=True)
for i, name in enumerate(rank, 1):
    print(f"{i}. {name:<26} {score(name):.3f}")
print()
print("换权重（安全优先：隔离 0.7）时排名会变 —— 打分只是把取舍显式化")

## §6 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | 容器与 VM 的唯一硬分界？ | 有没有独立内核 |
| 2 | gVisor 是虚拟机吗？ | 不是；共享内核 + 用户态内核（Sentry）翻译 syscall |
| 3 | Firecracker 为什么快？ | 砍设备模型（无 BIOS/PCI/图形）、极小代码基 |
| 4 | Wasm 能当通用沙箱吗？ | 不能；无 fork/exec、原生依赖跑不了 |
| 5 | 什么场景"进程级沙箱就够"？ | 单租户 + 代码基本可控 + 目标是防误操作 |
| 6 | 微 VM 的密度为什么低于容器？ | 每 VM 独立内核 + 不共享 page cache |

**相关长文**：[环节02-隔离强度谱系与选型.md](./环节02-隔离强度谱系与选型.md)